# Sample fur manuelle Re-Annotation

Dieses Notebook wahlt aus den annotierten Videos pro Gruppe zufallig eine Teilmenge aus und kopiert sie in einen neuen Ordner.

Standard: `25` Videos fur `ai` und `real` (falls vorhanden).

In [1]:
from pathlib import Path
from datetime import datetime
import shutil
import pandas as pd

# Konfiguration
METADATA_CSV = Path('../03_datasets/influencer_balanced/01_AI_AND_REAL_TIKTOK_VIDEOS_stratified_per_influencer_50.csv')
VIDEOS_DIR = Path('../02_media/stratified_sample/videos')
OUTPUT_BASE_DIR = Path('../06_human_evaluation_reannotation/reannotation_samples')

GROUP_COLUMN = 'influencer_type'
SAMPLES_PER_GROUP = 25
RANDOM_SEED = 42


# Bei abweichendem Schema die Video-ID-Spalte hier anpassen
VIDEO_ID_COLUMN = 'video_id'
VIDEO_EXTENSIONS = ['.mp4', '.mov', '.mkv', '.avi', '.webm']

In [3]:
if not METADATA_CSV.exists():
    raise FileNotFoundError(f'Metadaten-CSV nicht gefunden: {METADATA_CSV}')
if not VIDEOS_DIR.exists():
    raise FileNotFoundError(f'Video-Ordner nicht gefunden: {VIDEOS_DIR}')

df = pd.read_csv(METADATA_CSV)
df.columns = [c.strip() for c in df.columns]  # Leerzeichen in Headern entfernen
required_cols = {VIDEO_ID_COLUMN, GROUP_COLUMN}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f'Fehlende Spalten in CSV: {missing}')

df = df.copy()
target_groups = sorted(df[GROUP_COLUMN].dropna().astype(str).str.strip().unique().tolist())

# Videodateien nach Dateiname indexieren
video_files = []
for ext in VIDEO_EXTENSIONS:
    video_files.extend(VIDEOS_DIR.glob(f'*{ext}'))
video_map = {p.stem: p for p in video_files}

df['_video_stem'] = df[VIDEO_ID_COLUMN].astype(str).str.strip()
df['src_path'] = df['_video_stem'].map(video_map)
df_valid = df[df['src_path'].notna()].copy()

print(f'CSV-Zeilen insgesamt: {len(df)}')
print(f'Zeilen mit vorhandener Video-Datei: {len(df_valid)}')
print('Verfugbare Gruppen:', target_groups)

run_name = datetime.now().strftime('sample_%Y%m%d_%H%M%S')
output_dir = OUTPUT_BASE_DIR / run_name
output_dir.mkdir(parents=True, exist_ok=True)

selected_parts = []
for group in target_groups:
    gdf = df_valid[df_valid[GROUP_COLUMN].astype(str).str.strip() == group].copy()
    if gdf.empty:
        print(f'Warnung: Gruppe {group!r} ist leer und wird ubersprungen.')
        continue

    n = min(SAMPLES_PER_GROUP, len(gdf))
    if len(gdf) < SAMPLES_PER_GROUP:
        print(f'Warnung: Gruppe {group!r} hat nur {len(gdf)} Videos. Es werden {n} kopiert.')

    sampled = gdf.sample(n=n, random_state=RANDOM_SEED).copy()
    group_out = output_dir / group
    group_out.mkdir(parents=True, exist_ok=True)

    copied = 0
    dst_paths = []
    for _, row in sampled.iterrows():
        src = Path(row['src_path'])
        dst = group_out / src.name
        shutil.copy2(src, dst)
        copied += 1
        dst_paths.append(str(dst))

    sampled['_dst_path'] = dst_paths
    selected_parts.append(sampled)
    print(f'Gruppe {group!r}: {copied} Videos kopiert nach {group_out}')

if not selected_parts:
    raise RuntimeError('Es wurde keine Gruppe verarbeitet. Prufe die Daten.')

selected_df = pd.concat(selected_parts, ignore_index=True)
manifest_path = output_dir / 'selected_videos_manifest.csv'
selected_df.to_csv(manifest_path, index=False)

print('\nFertig.')
print(f'Zielordner: {output_dir}')
print(f'Manifest:   {manifest_path}')
selected_df[[VIDEO_ID_COLUMN, GROUP_COLUMN, 'src_path', '_dst_path']].head(10)

CSV-Zeilen insgesamt: 500
Zeilen mit vorhandener Video-Datei: 500
Verfugbare Gruppen: ['ai', 'real']
Gruppe 'ai': 25 Videos kopiert nach ..\data\reannotation_samples\sample_20260402_173603\ai
Gruppe 'real': 25 Videos kopiert nach ..\data\reannotation_samples\sample_20260402_173603\real

Fertig.
Zielordner: ..\data\reannotation_samples\sample_20260402_173603
Manifest:   ..\data\reannotation_samples\sample_20260402_173603\selected_videos_manifest.csv


,video_id,influencer_type,src_path,_dst_path
0,7032242293411286273,ai,..\02_media/stratified_sample/videos\703224...,..\data\reannotation_samples\sample_20260402_1...
1,7516241082754010414,ai,..\02_media/stratified_sample/videos\751624...,..\data\reannotation_samples\sample_20260402_1...
2,7465128290244742442,ai,..\02_media/stratified_sample/videos\746512...,..\data\reannotation_samples\sample_20260402_1...
3,7516564086872624397,ai,..\02_media/stratified_sample/videos\751656...,..\data\reannotation_samples\sample_20260402_1...
4,7265585005022203137,ai,..\02_media/stratified_sample/videos\726558...,..\data\reannotation_samples\sample_20260402_1...
5,7037583058626759941,ai,..\02_media/stratified_sample/videos\703758...,..\data\reannotation_samples\sample_20260402_1...
6,7014189798315298053,ai,..\02_media/stratified_sample/videos\701418...,..\data\reannotation_samples\sample_20260402_1...
7,7129589619099847982,ai,..\02_media/stratified_sample/videos\712958...,..\data\reannotation_samples\sample_20260402_1...
8,7516376664570416426,ai,..\02_media/stratified_sample/videos\751637...,..\data\reannotation_samples\sample_20260402_1...
9,7258222058293710088,ai,..\02_media/stratified_sample/videos\725822...,..\data\reannotation_samples\sample_20260402_1...


In [4]:
# Manifest nach KI und Real aufteilen
manifest_split_df = pd.read_csv(manifest_path)
manifest_split_df.columns = [c.strip() for c in manifest_split_df.columns]
legacy_map = {'video_stem': '_video_stem', 'src_path': 'src_path', 'dst_path': '_dst_path'}
manifest_split_df = manifest_split_df.rename(columns={k: v for k, v in legacy_map.items() if k in manifest_split_df.columns})

if GROUP_COLUMN not in manifest_split_df.columns:
    raise ValueError(f"Spalte '{GROUP_COLUMN}' nicht im Manifest gefunden: {manifest_path}")

group_series = manifest_split_df[GROUP_COLUMN].astype(str).str.strip().str.lower()
ai_df = manifest_split_df[group_series == 'ai'].copy()
real_df = manifest_split_df[group_series == 'real'].copy()

ai_out = output_dir / 'selected_videos_manifest_ai.csv'
real_out = output_dir / 'selected_videos_manifest_real.csv'

ai_df.to_csv(ai_out, index=False)
real_df.to_csv(real_out, index=False)

print(f'AI rows:   {len(ai_df)} -> {ai_out}')
print(f'Real rows: {len(real_df)} -> {real_out}')


AI rows:   25 -> ..\data\reannotation_samples\sample_20260402_173603\selected_videos_manifest_ai.csv
Real rows: 25 -> ..\data\reannotation_samples\sample_20260402_173603\selected_videos_manifest_real.csv


In [ ]:
# Fehlende Spalten ?ber video_id aus der Ergebnisdatei erg?nzen
RESULTS_CSV = Path('../04_analysis_results/visual_features/99_AI_AND_REAL_TIKTOK_VIDEOS_all_results_merged.csv')

if not RESULTS_CSV.exists():
    raise FileNotFoundError(f'Ergebnis-CSV nicht gefunden: {RESULTS_CSV}')

results_df = pd.read_csv(RESULTS_CSV)
results_df.columns = [c.strip() for c in results_df.columns]

if VIDEO_ID_COLUMN not in results_df.columns:
    raise ValueError(f"Spalte '{VIDEO_ID_COLUMN}' nicht in Ergebnis-CSV gefunden: {RESULTS_CSV}")

results_df = results_df.copy()
results_df[VIDEO_ID_COLUMN] = results_df[VIDEO_ID_COLUMN].astype(str).str.strip()
selected_df = selected_df.copy()
selected_df[VIDEO_ID_COLUMN] = selected_df[VIDEO_ID_COLUMN].astype(str).str.strip()

duplicate_ids = results_df.loc[results_df[VIDEO_ID_COLUMN].duplicated(), VIDEO_ID_COLUMN].unique().tolist()
if duplicate_ids:
    raise ValueError(
        'Die Ergebnis-CSV enthalt doppelte video_id-Werte. Eindeutiger Merge nicht moglich: '
        f'{duplicate_ids[:10]}'
    )

missing_columns = [column for column in results_df.columns if column not in selected_df.columns]
enrichment_columns = [VIDEO_ID_COLUMN] + [column for column in missing_columns if column != VIDEO_ID_COLUMN]
enrichment_df = results_df[enrichment_columns].copy()

selected_df_enriched = selected_df.merge(
    enrichment_df,
    on=VIDEO_ID_COLUMN,
    how='left',
    validate='m:1'
)

added_columns = [column for column in missing_columns if column != VIDEO_ID_COLUMN]
if added_columns:
    unmatched_ids = selected_df_enriched.loc[
        selected_df_enriched[added_columns].isna().all(axis=1),
        VIDEO_ID_COLUMN
    ].tolist()
else:
    unmatched_ids = []

enriched_manifest_path = output_dir / 'selected_videos_manifest_enriched.csv'
selected_df_enriched.to_csv(enriched_manifest_path, index=False)

enriched_group_series = selected_df_enriched[GROUP_COLUMN].astype(str).str.strip().str.lower()
ai_enriched_df = selected_df_enriched[enriched_group_series == 'ai'].copy()
real_enriched_df = selected_df_enriched[enriched_group_series == 'real'].copy()

ai_enriched_out = output_dir / 'selected_videos_manifest_ai_enriched.csv'
real_enriched_out = output_dir / 'selected_videos_manifest_real_enriched.csv'
ai_enriched_df.to_csv(ai_enriched_out, index=False)
real_enriched_df.to_csv(real_enriched_out, index=False)

print(f'Fehlende Spalten aus Master-CSV erganzt: {len(added_columns)}')
print(f'Enriched Manifest gespeichert unter: {enriched_manifest_path}')
print(f'AI enriched rows:   {len(ai_enriched_df)} -> {ai_enriched_out}')
print(f'Real enriched rows: {len(real_enriched_df)} -> {real_enriched_out}')
if unmatched_ids:
    print('Warnung: Fur diese video_ids wurden keine Master-Daten gefunden:')
    print(unmatched_ids)

selected_df_enriched.head(5)


In [ ]:
# Numerische Annotationsspalten auswerten und interpretierbar skalieren
ANNOTATION_EXCLUDE_SUFFIXES = (
    '__has_frames',
    '__has_video',
    '__frames_scanned',
    '__video_fps',
    '__video_duration_seconds_est',
    '__processed_frame_count',
    '__processed_frame_pairs',
    '__detected_face_frames',
    '__detected_skin_face_frames',
    '__detected_emotion_frames',
    '__detected_pose_frames',
    '__detected_person_frames',
    '__sampled_frames_personen_anzahl',
    '__aesthetic_quality_scored_frames',
    '__beauty_detected_face_frames',
    '__camera_distance_unique_labels',
    '__scene_unique_labels',
    '__emotion_unique_labels',
)

LIKERT_LABELS = [
    '1 = sehr niedrig',
    '2 = niedrig',
    '3 = mittel',
    '4 = hoch',
    '5 = sehr hoch',
]

def is_binary_series(series):
    values = {str(v).strip().lower() for v in series.dropna().unique()}
    return values.issubset({'0', '1', '0.0', '1.0', 'true', 'false'})

def build_likert_edges(min_value, max_value, steps=5):
    if pd.isna(min_value) or pd.isna(max_value):
        return None
    if min_value == max_value:
        return [float(min_value)] * (steps + 1)
    width = (max_value - min_value) / steps
    return [round(float(min_value + width * i), 4) for i in range(steps + 1)]

def likert_from_range(value, edges):
    if pd.isna(value) or edges is None:
        return pd.NA
    if edges[0] == edges[-1]:
        return 3
    for idx in range(1, len(edges)):
        if value <= edges[idx] or idx == len(edges) - 1:
            return idx
    return len(edges) - 1

def format_range(min_value, max_value):
    if pd.isna(min_value) or pd.isna(max_value):
        return 'keine numerischen Werte'
    return f'{min_value:.4f} bis {max_value:.4f}'

numeric_annotation_columns = []
for column in selected_df_enriched.columns:
    if '__' not in column or column not in results_df.columns:
        continue
    if column.endswith(ANNOTATION_EXCLUDE_SUFFIXES):
        continue

    source_series = results_df[column]
    if is_binary_series(source_series):
        continue

    project_numeric = pd.to_numeric(source_series, errors='coerce')
    sample_numeric = pd.to_numeric(selected_df_enriched[column], errors='coerce')
    if project_numeric.notna().sum() == 0 and sample_numeric.notna().sum() == 0:
        continue

    numeric_annotation_columns.append(column)

range_rows = []
for column in numeric_annotation_columns:
    project_numeric = pd.to_numeric(results_df[column], errors='coerce')
    sample_numeric = pd.to_numeric(selected_df_enriched[column], errors='coerce')

    project_min = project_numeric.min()
    project_max = project_numeric.max()
    sample_min = sample_numeric.min()
    sample_max = sample_numeric.max()
    likert_edges = build_likert_edges(project_min, project_max, steps=5)

    likert_column = f'{column}__likert_5'
    label_column = f'{column}__likert_label'
    selected_df_enriched[likert_column] = sample_numeric.apply(lambda value: likert_from_range(value, likert_edges)).astype('Int64')
    selected_df_enriched[label_column] = selected_df_enriched[likert_column].apply(
        lambda value: LIKERT_LABELS[int(value) - 1] if pd.notna(value) else pd.NA
    )

    range_rows.append({
        'column': column,
        'model_min': project_min,
        'model_max': project_max,
        'sample_min': sample_min,
        'sample_max': sample_max,
        'model_range': format_range(project_min, project_max),
        'sample_range': format_range(sample_min, sample_max),
        'interpretation_scale': ' | '.join(LIKERT_LABELS),
        'scale_edges_project_range': likert_edges,
    })

ranges_df = pd.DataFrame(range_rows).sort_values('column').reset_index(drop=True)
ranges_path = output_dir / 'selected_videos_numeric_annotation_ranges.csv'
selected_df_enriched.to_csv(enriched_manifest_path, index=False)
ranges_df.to_csv(ranges_path, index=False)

print('Numerische Annotationsspalten mit Projekt- und Sample-Range:')
for _, row in ranges_df.iterrows():
    print(
        f"- {row['column']}: Modell/Projekt {row['model_range']}; "
        f"Sample {row['sample_range']}; "
        f"Likert-Skala: {row['interpretation_scale']}; "
        f"Grenzen: {row['scale_edges_project_range']}"
    )

print(f'Range-Ubersicht gespeichert unter: {ranges_path}')
ranges_df
